# 📄 Production-Level RAG System — Warren Buffett 2024 Shareholder Letter

This notebook builds a **Retrieval-Augmented Generation (RAG)** pipeline from scratch.

The system allows you to:
- Load a **PDF** (Warren Buffett's 2024 Shareholder Letter)
- Ask **explanatory questions** about its content
- Get answers **grounded in retrieved document chunks** using a local LLM

We intentionally avoid frameworks like LangChain or LlamaIndex to fully understand every component.

---

## 🗺️ Pipeline Overview

### Two Phases
1. **Offline Indexing** — Process the document once and store embeddings
2. **Online Query Serving** — Retrieve relevant chunks and generate grounded answers

### Full Pipeline Steps
1. Load and parse the PDF
2. EDA — analyze text statistics
3. Chunk the text (2 strategies: Fixed-Size + Sentence-Aware)
4. Generate embeddings (Open-Source + Closed-Source)
5. Index with FAISS
6. Retrieve top-k chunks
7. Build grounded prompt
8. Generate answer with Gemma-7B-IT
9. Evaluate retrieval quality

---

## ⚙️ Setup Requirements
- Google Colab Pro (A100 GPU)
- Hugging Face token (for Gemma access)
- Cohere API key (for closed-source embeddings)
- Warren Buffett 2024 Letter PDF uploaded to Colab

## Section 1 — Install Dependencies

In [ ]:
# -----------------------------------------------
# Install all required libraries
# -----------------------------------------------
!pip install -q PyMuPDF
!pip install -q spacy
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q cohere
!pip install -q transformers accelerate bitsandbytes
!pip install -q tqdm pandas numpy
!pip install -q ragas datasets openai langchain-openai nest-asyncio

# Download spaCy English model
!python -m spacy download en_core_web_sm

print("[INFO] All dependencies installed successfully.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 70.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
[INFO] All dependencies installed successfully.


## Section 2 — Configuration & API Keys



In [ ]:
# -----------------------------------------------
# Configuration — paste your keys here
# -----------------------------------------------
import os

# Hugging Face token (for Gemma-7B-IT access)
HF_TOKEN = "paste_your_huggingface_token_here"

# Cohere API key (for closed-source embeddings)
COHERE_API_KEY = "paste_your_cohere_api_key_here"

# Path to your uploaded PDF
# Upload the Buffett letter to Colab using the file browser on the left
PDF_PATH = "/content/drive/MyDrive/Colab Notebooks/Berkshire Hathaway Inc .pdf"

# Set environment variables
os.environ["COHERE_API_KEY"] = COHERE_API_KEY

print("[INFO] Configuration set.")
print(f"[INFO] PDF path: {PDF_PATH}")

[INFO] Configuration set.
[INFO] PDF path: /content/drive/MyDrive/Colab Notebooks/Berkshire Hathaway Inc .pdf


## Section 3 — Login to Hugging Face

In [ ]:
# -----------------------------------------------
# Authenticate with Hugging Face
# Required to download Gemma-7B-IT
# -----------------------------------------------
from huggingface_hub import login

login(token=HF_TOKEN)
print("[INFO] Hugging Face login successful.")

[INFO] Hugging Face login successful.


## Section 4 — Phase 1: Document Loading & Text Extraction

We use **PyMuPDF (fitz)** to extract text page by page from the PDF.

### Why this matters
- PDFs contain unstructured layouts
- Extraction quality directly affects chunking, embedding, and retrieval
- We store metadata (page number, word count, token count) alongside text

In [ ]:
# -----------------------------------------------
# Step 1: Extract text from PDF page by page
# -----------------------------------------------
import fitz  # PyMuPDF
from tqdm.auto import tqdm


def preprocess_text(raw_text: str) -> str:
    """
    Lightweight text cleaning:
    - Replace newlines with spaces
    - Strip leading/trailing whitespace
    """
    return raw_text.replace("\n", " ").strip()


def extract_pdf_text(pdf_path: str) -> list[dict]:
    """
    Extracts and preprocesses text from each page of a PDF.

    Returns a list of page records with:
    - page_index: zero-based page number
    - text: cleaned page text
    - char_count: character count
    - word_count: word count
    - token_count_approx: approximate token count (chars / 4)
    """
    document = fitz.open(pdf_path)
    pages = []

    for page_index, page in tqdm(
        enumerate(document),
        total=len(document),
        desc="Extracting pages"
    ):
        raw_text = page.get_text("text")
        cleaned_text = preprocess_text(raw_text)

        pages.append({
            "page_index": page_index,
            "char_count": len(cleaned_text),
            "word_count": len(cleaned_text.split()),
            "token_count_approx": len(cleaned_text) / 4,
            "text": cleaned_text,
        })

    document.close()
    print(f"[INFO] Extracted {len(pages)} pages from PDF.")
    return pages


# Run extraction
pdf_pages = extract_pdf_text(PDF_PATH)

# Preview first page
print("\n--- Page 1 Preview ---")
print(f"Words: {pdf_pages[0]['word_count']}")
print(f"Tokens (approx): {pdf_pages[0]['token_count_approx']:.0f}")
print(f"Text snippet: {pdf_pages[0]['text'][:300]}")

Extracting pages:   0%|          | 0/15 [00:00<?, ?it/s]

[INFO] Extracted 15 pages from PDF.

--- Page 1 Preview ---
Words: 366
Tokens (approx): 547
Text snippet: BERKSHIRE HATHAWAY INC.  To the Shareholders of Berkshire Hathaway Inc.:  This letter comes to you as part of Berkshire’s annual report. As a public company, we  are required to periodically tell you many specific facts and figures.  “Report,” however, implies a greater responsibility. In addition t


## Section 5 — EDA: Text Statistics

Before chunking, we analyze our document to make informed design decisions.

### Why EDA matters in RAG
- Average token count per page informs chunk size
- If average tokens > embedding model limit, we must chunk more aggressively
- Token distribution reveals whether pages are uniform or highly variable

In [ ]:
# -----------------------------------------------
# Step 2: EDA — analyze text statistics
# -----------------------------------------------
import pandas as pd

# Create DataFrame from extracted pages
stats_df = pd.DataFrame(pdf_pages)

print("=" * 60)
print("DOCUMENT STATISTICS")
print("=" * 60)
print(f"Total pages          : {len(pdf_pages)}")
print(f"Total words          : {stats_df['word_count'].sum():,}")
print(f"Total tokens (approx): {stats_df['token_count_approx'].sum():,.0f}")
print(f"Avg words/page       : {stats_df['word_count'].mean():.1f}")
print(f"Avg tokens/page      : {stats_df['token_count_approx'].mean():.1f}")
print(f"Max tokens on a page : {stats_df['token_count_approx'].max():.0f}")
print(f"Min tokens on a page : {stats_df['token_count_approx'].min():.0f}")
print("=" * 60)

print("\nNote: all-mpnet-base-v2 has a 384-token limit.")
print("Chunks must stay below this to avoid silent truncation.")

# Show full stats table
stats_df[["page_index", "word_count", "token_count_approx"]].head(10)

DOCUMENT STATISTICS
Total pages          : 15
Total words          : 8,916
Total tokens (approx): 10,171
Avg words/page       : 594.4
Avg tokens/page      : 678.1
Max tokens on a page : 954
Min tokens on a page : 491

Note: all-mpnet-base-v2 has a 384-token limit.
Chunks must stay below this to avoid silent truncation.


,page_index,word_count,token_count_approx
0,0,366,547.25
1,1,505,718.00
2,2,382,536.25
3,3,383,596.25
4,4,532,653.50
5,5,431,654.25
6,6,504,773.00
7,7,485,717.00
8,8,442,680.50
9,9,383,595.25


## Section 6 — Phase 1: Chunking Strategy 1 — Fixed-Size with Overlap

### What is Fixed-Size Chunking?
Documents are split into chunks of exactly **N characters**, regardless of sentence boundaries.
An **overlap** of M characters is shared between consecutive chunks to prevent information loss at boundaries.

### Sliding Window Math
- Chunk size: **500 characters**
- Overlap: **100 characters** (20%)
- Chunk 1: chars 0–500
- Chunk 2: chars 400–900
- Chunk k: starts at (k × 400)

### Advantages
- Simple and fast
- Predictable chunk sizes
- Good baseline for comparison

### Disadvantages
- May split sentences mid-stream
- No awareness of topic or sentence boundaries

In [ ]:
# -----------------------------------------------
# Chunking Strategy 1: Fixed-Size with Overlap
# -----------------------------------------------
import re

CHUNK_SIZE = 1000    # characters per chunk
CHUNK_OVERLAP = 200   # characters of overlap between consecutive chunks


def fixed_size_chunks(text: str, size: int, overlap: int) -> list[str]:
    """
    Splits text into fixed-size character chunks with overlap.

    Parameters
    ----------
    text    : full document or page text
    size    : number of characters per chunk
    overlap : number of overlapping characters between chunks

    Returns
    -------
    List of text chunk strings
    """
    step = size - overlap
    chunks = []
    for start in range(0, len(text), step):
        chunk = text[start:start + size].strip()
        if chunk:
            chunks.append(chunk)
    return chunks


def build_fixed_chunk_records(pages: list[dict], size: int, overlap: int) -> list[dict]:
    """
    Applies fixed-size chunking to all pages and returns
    a flat list of chunk records with metadata.
    """
    records = []
    for page in tqdm(pages, desc="Fixed-size chunking"):
        chunks = fixed_size_chunks(page["text"], size, overlap)
        for i, chunk_text in enumerate(chunks):
            # Clean up spacing
            chunk_text = re.sub(r"\.([A-Z])", r". \1", chunk_text)
            records.append({
                "page_index": page["page_index"],
                "chunk_index": i,
                "strategy": "fixed_size",
                "text": chunk_text,
                "char_count": len(chunk_text),
                "word_count": len(chunk_text.split()),
                "token_count_approx": len(chunk_text) / 4,
            })
    return records


# Build fixed-size chunks
fixed_chunks = build_fixed_chunk_records(pdf_pages, CHUNK_SIZE, CHUNK_OVERLAP)

# Filter out very small chunks (< 30 tokens)
fixed_chunks = [c for c in fixed_chunks if c["token_count_approx"] > 30]

print(f"[INFO] Fixed-size chunks created : {len(fixed_chunks)}")
print(f"[INFO] Avg tokens per chunk      : {sum(c['token_count_approx'] for c in fixed_chunks) / len(fixed_chunks):.1f}")
print("\n--- Sample Fixed-Size Chunk ---")
print(fixed_chunks[0]["text"])

Fixed-size chunking:   0%|          | 0/15 [00:00<?, ?it/s]

[INFO] Fixed-size chunks created : 57
[INFO] Avg tokens per chunk      : 215.1

--- Sample Fixed-Size Chunk ---
BERKSHIRE HATHAWAY INC.  To the Shareholders of Berkshire Hathaway Inc.:  This letter comes to you as part of Berkshire’s annual report. As a public company, we  are required to periodically tell you many specific facts and figures.  “Report,” however, implies a greater responsibility. In addition to the mandated data, we  believe we owe you additional commentary about what you own and how we think. Our goal is  to communicate with you in a manner that we would wish you to use if our positions were  reversed – that is, if you were Berkshire’s CEO while I and my family were passive investors,  trusting you with our savings.  This approach leads us to an annual recitation of both good and bad developments at the  many businesses you indirectly own through your Berkshire shares. When discussing problems  at specific subsidiaries, we do, however, try to follow the advice Tom Murp

## Section 7 — Phase 1: Chunking Strategy 2 — Sentence-Aware Chunking

### What is Sentence-Aware Chunking?
Instead of cutting at fixed character positions, we use **spaCy** to detect sentence boundaries,
then group every **N sentences** into one chunk.

### Why it's better semantically
- Chunks never split mid-sentence
- Each chunk contains complete, coherent thoughts
- Better semantic signal for embedding models

### Parameters
- **5 sentences per chunk** — balances coherence and token limits

### Disadvantages
- Slower than fixed-size (requires NLP processing)
- Chunk sizes vary (some sentences are longer than others)

In [ ]:
# -----------------------------------------------
# Chunking Strategy 2: Sentence-Aware Chunking
# -----------------------------------------------
import spacy

# Load spaCy English model
nlp = spacy.load("en_core_web_sm")

SENTENCES_PER_CHUNK = 10


def get_sentences(text: str) -> list[str]:
    """
    Uses spaCy to split text into sentences.
    More accurate than splitting on periods alone.
    """
    doc = nlp(text)
    return [str(sent).strip() for sent in doc.sents if str(sent).strip()]


def group_sentences(sentences: list[str], group_size: int) -> list[str]:
    """
    Groups a list of sentences into chunks of group_size sentences.
    Example: 12 sentences with group_size=5 → [[s1..s5], [s6..s10], [s11..s12]]
    """
    return [
        " ".join(sentences[i:i + group_size])
        for i in range(0, len(sentences), group_size)
    ]


def build_sentence_chunk_records(pages: list[dict], group_size: int) -> list[dict]:
    """
    Applies sentence-aware chunking to all pages and returns
    a flat list of chunk records with metadata.
    """
    records = []
    for page in tqdm(pages, desc="Sentence-aware chunking"):
        if not page["text"].strip():
            continue
        sentences = get_sentences(page["text"])
        chunks = group_sentences(sentences, group_size)
        for i, chunk_text in enumerate(chunks):
            records.append({
                "page_index": page["page_index"],
                "chunk_index": i,
                "strategy": "sentence_aware",
                "text": chunk_text,
                "char_count": len(chunk_text),
                "word_count": len(chunk_text.split()),
                "token_count_approx": len(chunk_text) / 4,
            })
    return records


# Build sentence-aware chunks
sentence_chunks = build_sentence_chunk_records(pdf_pages, SENTENCES_PER_CHUNK)

# Filter out very small chunks (< 30 tokens)
sentence_chunks = [c for c in sentence_chunks if c["token_count_approx"] > 30]

print(f"[INFO] Sentence-aware chunks created : {len(sentence_chunks)}")
print(f"[INFO] Avg tokens per chunk          : {sum(c['token_count_approx'] for c in sentence_chunks) / len(sentence_chunks):.1f}")
print("\n--- Sample Sentence-Aware Chunk ---")
print(sentence_chunks[0]["text"])

Sentence-aware chunking:   0%|          | 0/15 [00:00<?, ?it/s]

[INFO] Sentence-aware chunks created : 57
[INFO] Avg tokens per chunk          : 177.7

--- Sample Sentence-Aware Chunk ---
BERKSHIRE HATHAWAY INC.  To the Shareholders of Berkshire Hathaway Inc.:  This letter comes to you as part of Berkshire’s annual report. As a public company, we  are required to periodically tell you many specific facts and figures. “Report,” however, implies a greater responsibility. In addition to the mandated data, we  believe we owe you additional commentary about what you own and how we think. Our goal is  to communicate with you in a manner that we would wish you to use if our positions were  reversed – that is, if you were Berkshire’s CEO while I and my family were passive investors,  trusting you with our savings. This approach leads us to an annual recitation of both good and bad developments at the  many businesses you indirectly own through your Berkshire shares. When discussing problems  at specific subsidiaries, we do, however, try to follow the advic

## Section 8 — Chunking Strategy Comparison

Before embedding, let's compare the two strategies side by side.


In [ ]:
# -----------------------------------------------
# Compare chunking strategies
# -----------------------------------------------
fixed_df = pd.DataFrame(fixed_chunks)
sentence_df = pd.DataFrame(sentence_chunks)

print("=" * 60)
print("CHUNKING STRATEGY COMPARISON")
print("=" * 60)
print(f"{'Metric':<30} {'Fixed-Size':>15} {'Sentence-Aware':>15}")
print("-" * 60)
print(f"{'Total chunks':<30} {len(fixed_chunks):>15} {len(sentence_chunks):>15}")
print(f"{'Avg tokens/chunk':<30} {fixed_df['token_count_approx'].mean():>15.1f} {sentence_df['token_count_approx'].mean():>15.1f}")
print(f"{'Min tokens/chunk':<30} {fixed_df['token_count_approx'].min():>15.1f} {sentence_df['token_count_approx'].min():>15.1f}")
print(f"{'Max tokens/chunk':<30} {fixed_df['token_count_approx'].max():>15.1f} {sentence_df['token_count_approx'].max():>15.1f}")
print(f"{'Std dev tokens':<30} {fixed_df['token_count_approx'].std():>15.1f} {sentence_df['token_count_approx'].std():>15.1f}")
print("=" * 60)
print("\nNote: all-mpnet-base-v2 limit = 384 tokens.")
print("Chunks exceeding this will be silently truncated.")

CHUNKING STRATEGY COMPARISON
Metric                              Fixed-Size  Sentence-Aware
------------------------------------------------------------
Total chunks                                57              57
Avg tokens/chunk                         215.1           177.7
Min tokens/chunk                          43.0            46.0
Max tokens/chunk                         250.5           342.8
Std dev tokens                            63.7            79.3

Note: all-mpnet-base-v2 limit = 384 tokens.
Chunks exceeding this will be silently truncated.


## Section 9 — Phase 2: Open-Source Embeddings (all-mpnet-base-v2)

### What are embeddings?
An embedding converts text into a dense numerical vector that captures semantic meaning.
Similar meaning → similar vectors → close together in vector space.

### Model: all-mpnet-base-v2
- Open-source, free to use locally
- Max input: **384 tokens**
- Output: **768-dimensional vector**
- Strong performance for general retrieval tasks

### Critical Rule
We MUST use the same embedding model for both indexing AND querying.
Different models produce incompatible vector spaces.

In [ ]:
# -----------------------------------------------
# Open-Source Embeddings: all-mpnet-base-v2
# -----------------------------------------------
import torch
import numpy as np
from sentence_transformers import SentenceTransformer

# Auto-detect device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[INFO] Using device: {device}")

# Load open-source embedding model
print("[INFO] Loading open-source embedding model...")
os_embedding_model = SentenceTransformer(
    model_name_or_path="all-mpnet-base-v2",
    device=device
)
print("[INFO] Model loaded. Embedding dimension: 768")


def generate_embeddings_opensource(chunk_records: list[dict],
                                    model: SentenceTransformer,
                                    batch_size: int = 32) -> np.ndarray:
    """
    Generates embeddings for a list of chunk records using
    the open-source sentence-transformers model.

    Returns a numpy array of shape (n_chunks, embedding_dim)
    """
    texts = [chunk["text"] for chunk in chunk_records]
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True
    )
    return embeddings


# Generate open-source embeddings for BOTH chunking strategies
print("\n[INFO] Embedding fixed-size chunks (open-source)...")
fixed_os_embeddings = generate_embeddings_opensource(fixed_chunks, os_embedding_model)

print("\n[INFO] Embedding sentence-aware chunks (open-source)...")
sentence_os_embeddings = generate_embeddings_opensource(sentence_chunks, os_embedding_model)

print(f"\n[INFO] Fixed-size embeddings shape   : {fixed_os_embeddings.shape}")
print(f"[INFO] Sentence-aware embeddings shape: {sentence_os_embeddings.shape}")

[INFO] Using device: cuda
[INFO] Loading open-source embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[INFO] Model loaded. Embedding dimension: 768

[INFO] Embedding fixed-size chunks (open-source)...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]


[INFO] Embedding sentence-aware chunks (open-source)...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]


[INFO] Fixed-size embeddings shape   : (57, 768)
[INFO] Sentence-aware embeddings shape: (57, 768)


## Section 10 — Phase 2: Closed-Source Embeddings (Cohere)

### Model: Cohere embed-english-v3.0
- Closed-source API (requires API key)
- Output: **1024-dimensional vector**
- State-of-the-art English retrieval performance
- Requires specifying input_type: `search_document` for indexing, `search_query` for queries

### Comparison with Open-Source
| Feature | all-mpnet-base-v2 | Cohere embed-v3 |
|---|---|---|
| Cost | Free | Free trial |
| Privacy | Local | Sent to API |
| Dimensions | 768 | 1024 |
| Quality | Very good | Excellent |

In [ ]:
# -----------------------------------------------
# Closed-Source Embeddings: Cohere embed-english-v3.0
# -----------------------------------------------
import cohere
import time

# Initialize Cohere client
co = cohere.Client(api_key=COHERE_API_KEY)
print("[INFO] Cohere client initialized.")


def generate_embeddings_cohere(chunk_records: list[dict],
                                client: cohere.Client,
                                input_type: str = "search_document",
                                batch_size: int = 90) -> np.ndarray:
    """
    Generates embeddings for a list of chunk records using
    Cohere's embed-english-v3.0 API.

    Parameters
    ----------
    chunk_records : list of chunk dicts with 'text' key
    client        : initialized Cohere client
    input_type    : 'search_document' for indexing, 'search_query' for queries
    batch_size    : Cohere allows max 96 texts per call

    Returns a numpy array of shape (n_chunks, 1024)
    """
    texts = [chunk["text"] for chunk in chunk_records]
    all_embeddings = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Cohere embeddings"):
        batch = texts[i:i + batch_size]
        response = client.embed(
            texts=batch,
            model="embed-english-v3.0",
            input_type=input_type
        )
        all_embeddings.extend(response.embeddings)
        time.sleep(0.5)  # Respect rate limits

    return np.array(all_embeddings)


# Generate Cohere embeddings for BOTH chunking strategies
print("\n[INFO] Embedding fixed-size chunks (Cohere)...")
fixed_cohere_embeddings = generate_embeddings_cohere(fixed_chunks, co)

print("\n[INFO] Embedding sentence-aware chunks (Cohere)...")
sentence_cohere_embeddings = generate_embeddings_cohere(sentence_chunks, co)

print(f"\n[INFO] Fixed-size Cohere embeddings shape   : {fixed_cohere_embeddings.shape}")
print(f"[INFO] Sentence-aware Cohere embeddings shape: {sentence_cohere_embeddings.shape}")

[INFO] Cohere client initialized.

[INFO] Embedding fixed-size chunks (Cohere)...


Cohere embeddings:   0%|          | 0/1 [00:00<?, ?it/s]


[INFO] Embedding sentence-aware chunks (Cohere)...


Cohere embeddings:   0%|          | 0/1 [00:00<?, ?it/s]


[INFO] Fixed-size Cohere embeddings shape   : (57, 1024)
[INFO] Sentence-aware Cohere embeddings shape: (57, 1024)


## Section 11 — Phase 2: FAISS Vector Indexing

### What is FAISS?
FAISS (Facebook AI Similarity Search) is an open-source library for efficient similarity search over dense vectors.

### Why FAISS?
- Brute-force search works for small datasets but becomes slow at scale
- FAISS uses optimized index structures for fast nearest-neighbor search
- We use `IndexFlatIP` (Inner Product / dot product similarity)

### We build 4 indexes total
1. Fixed-size chunks + Open-source embeddings
2. Fixed-size chunks + Cohere embeddings
3. Sentence-aware chunks + Open-source embeddings
4. Sentence-aware chunks + Cohere embeddings

In [ ]:
# -----------------------------------------------
# FAISS Vector Indexing — Build 4 indexes
# -----------------------------------------------
import faiss


def build_faiss_index(embeddings: np.ndarray) -> faiss.IndexFlatIP:
    """
    Builds a FAISS flat inner-product index from embeddings.

    Parameters
    ----------
    embeddings : numpy array of shape (n_chunks, embedding_dim)

    Returns
    -------
    FAISS index ready for similarity search
    """
    # Normalize embeddings for cosine similarity
    faiss.normalize_L2(embeddings)

    # Build index
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)  # Inner product = cosine sim after normalization
    index.add(embeddings)

    return index


# Build all 4 FAISS indexes
print("[INFO] Building FAISS indexes...")

index_fixed_os = build_faiss_index(fixed_os_embeddings.copy().astype("float32"))
print(f"[INFO] Index 1 — Fixed + Open-source   : {index_fixed_os.ntotal} vectors")

index_fixed_cohere = build_faiss_index(fixed_cohere_embeddings.copy().astype("float32"))
print(f"[INFO] Index 2 — Fixed + Cohere        : {index_fixed_cohere.ntotal} vectors")

index_sentence_os = build_faiss_index(sentence_os_embeddings.copy().astype("float32"))
print(f"[INFO] Index 3 — Sentence + Open-source: {index_sentence_os.ntotal} vectors")

index_sentence_cohere = build_faiss_index(sentence_cohere_embeddings.copy().astype("float32"))
print(f"[INFO] Index 4 — Sentence + Cohere     : {index_sentence_cohere.ntotal} vectors")

print("\n[INFO] All 4 FAISS indexes built successfully.")

[INFO] Building FAISS indexes...
[INFO] Index 1 — Fixed + Open-source   : 57 vectors
[INFO] Index 2 — Fixed + Cohere        : 57 vectors
[INFO] Index 3 — Sentence + Open-source: 57 vectors
[INFO] Index 4 — Sentence + Cohere     : 57 vectors

[INFO] All 4 FAISS indexes built successfully.


## Section 12 — Phase 2: Retrieval Functions

### How Retrieval Works
1. Embed the user query using the **same model** used for indexing
2. Search the FAISS index for the top-k most similar chunk vectors
3. Return the matching chunk texts and similarity scores

### Top-K
We retrieve the **top 5 chunks** by default.
More chunks = more context but also more noise.

In [ ]:
# -----------------------------------------------
# Retrieval Functions
# -----------------------------------------------
import textwrap
from time import perf_counter as timer


def retrieve_opensource(
    query: str,
    index: faiss.IndexFlatIP,
    chunk_records: list[dict],
    model: SentenceTransformer,
    k: int = 5
) -> list[dict]:
    """
    Retrieves top-k relevant chunks using open-source embeddings.

    Returns list of chunk dicts with added 'score' field.
    """
    # Embed query
    query_vector = model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")
    faiss.normalize_L2(query_vector)

    # Search
    start = timer()
    scores, indices = index.search(query_vector, k)
    elapsed = timer() - start

    print(f"[INFO] Retrieved {k} chunks in {elapsed:.5f} seconds")

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < len(chunk_records):
            result = chunk_records[idx].copy()
            result["score"] = float(score)
            results.append(result)

    return results


def retrieve_cohere(
    query: str,
    index: faiss.IndexFlatIP,
    chunk_records: list[dict],
    client: cohere.Client,
    k: int = 5
) -> list[dict]:
    """
    Retrieves top-k relevant chunks using Cohere embeddings.

    Returns list of chunk dicts with added 'score' field.
    """
    # Embed query using Cohere — use 'search_query' input type
    response = client.embed(
        texts=[query],
        model="embed-english-v3.0",
        input_type="search_query"
    )
    query_vector = np.array(response.embeddings).astype("float32")
    faiss.normalize_L2(query_vector)

    # Search
    start = timer()
    scores, indices = index.search(query_vector, k)
    elapsed = timer() - start

    print(f"[INFO] Retrieved {k} chunks in {elapsed:.5f} seconds")

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < len(chunk_records):
            result = chunk_records[idx].copy()
            result["score"] = float(score)
            results.append(result)

    return results


def print_results(query: str, results: list[dict], width: int = 80):
    """Pretty-prints retrieval results."""
    print(f"\nQuery: '{query}'")
    print("=" * width)
    for i, result in enumerate(results, 1):
        print(f"\nResult {i} | Page {result['page_index']} | Score: {result['score']:.4f}")
        print("-" * width)
        print(textwrap.fill(result["text"], width))
    print("=" * width)


print("[INFO] Retrieval functions defined.")

[INFO] Retrieval functions defined.


## Section 13 — Test Retrieval: All 4 Index Combinations



In [ ]:
# -----------------------------------------------
# Test Retrieval — Sample Query
# -----------------------------------------------
TEST_QUERY = "Why does Buffett prefer long-term investments over short-term trading?"

print("\n" + "#" * 80)
print("INDEX 1: Fixed-Size Chunks + Open-Source Embeddings")
print("#" * 80)
results_fixed_os = retrieve_opensource(
    TEST_QUERY, index_fixed_os, fixed_chunks, os_embedding_model, k=3
)
print_results(TEST_QUERY, results_fixed_os)

print("\n" + "#" * 80)
print("INDEX 2: Fixed-Size Chunks + Cohere Embeddings")
print("#" * 80)
results_fixed_cohere = retrieve_cohere(
    TEST_QUERY, index_fixed_cohere, fixed_chunks, co, k=3
)
print_results(TEST_QUERY, results_fixed_cohere)

print("\n" + "#" * 80)
print("INDEX 3: Sentence-Aware Chunks + Open-Source Embeddings")
print("#" * 80)
results_sentence_os = retrieve_opensource(
    TEST_QUERY, index_sentence_os, sentence_chunks, os_embedding_model, k=3
)
print_results(TEST_QUERY, results_sentence_os)

print("\n" + "#" * 80)
print("INDEX 4: Sentence-Aware Chunks + Cohere Embeddings")
print("#" * 80)
results_sentence_cohere = retrieve_cohere(
    TEST_QUERY, index_sentence_cohere, sentence_chunks, co, k=3
)
print_results(TEST_QUERY, results_sentence_cohere)


################################################################################
INDEX 1: Fixed-Size Chunks + Open-Source Embeddings
################################################################################
[INFO] Retrieved 3 chunks in 0.00007 seconds

Query: 'Why does Buffett prefer long-term investments over short-term trading?'

Result 1 | Page 6 | Score: 0.4513
--------------------------------------------------------------------------------
With marketable equities, it is easier to change course when I make a mistake.
Berkshire’s present size, it should be underscored, diminishes this valuable
option. We can’t  come and go on a dime. Sometimes a year or more is required to
establish or divest an  investment. Additionally, with ownership of minority
positions we can’t change management if  that action is needed or control what
is done with capital flows if we are unhappy with the  decisions being made.
With controlled companies, we can dictate these decisions, but we have fa

## Section 14 — Load Gemma-7B-IT (Local LLM)

### Why Gemma-7B-IT?
- Instruction-tuned — designed to follow prompts and answer questions
- Strong reasoning for its size
- Runs locally — data never leaves our Colab session
- Free via Hugging Face

### Hardware
- Running on A100 (Colab Pro) in **Float16** — no quantization needed
- Uses **Flash Attention 2** for faster inference on A100

In [ ]:
# -----------------------------------------------
# Load Gemma-7B-IT
# -----------------------------------------------
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from transformers.utils import is_flash_attn_2_available

MODEL_ID = "google/gemma-7b-it"

print(f"[INFO] Loading model: {MODEL_ID}")
print(f"[INFO] Device: {device}")

# Select attention implementation
if (
    device == "cuda"
    and is_flash_attn_2_available()
    and torch.cuda.get_device_capability(0)[0] >= 8
):
    attn_implementation = "flash_attention_2"
else:
    attn_implementation = "sdpa"

print(f"[INFO] Attention: {attn_implementation}")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

# Load model
llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    attn_implementation=attn_implementation,
    device_map="auto",
    low_cpu_mem_usage=True
)

print(f"[INFO] Model loaded successfully.")
total_params = sum(p.numel() for p in llm_model.parameters())
print(f"[INFO] Total parameters: {total_params:,}")

[INFO] Loading model: google/gemma-7b-it
[INFO] Device: cuda
[INFO] Attention: sdpa


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

[INFO] Model loaded successfully.
[INFO] Total parameters: 8,537,680,896


## Section 15 — Prompt Construction & Grounded Generation

### What is Grounding?
Grounding means forcing the LLM to answer **only using the retrieved chunks**.
This is achieved through prompt engineering — explicit instructions in the prompt.

### Prompt Template
```
Based on the context below, answer the user query.
Use ONLY the provided context.
Do NOT make up information.
If the answer is not in the context, say "I don't know."

Context:
[Retrieved chunks]

User Query:
[Question]

Answer:
```

In [ ]:
# -----------------------------------------------
# Prompt Construction
# -----------------------------------------------

def build_prompt(query: str, context_items: list[dict]) -> str:
    """
    Builds a grounded RAG prompt by injecting retrieved
    chunks into the prompt template.

    Parameters
    ----------
    query        : user's natural language question
    context_items: list of retrieved chunk dicts

    Returns
    -------
    Formatted prompt string
    """
    # Format context blocks
    context_blocks = []
    for i, item in enumerate(context_items, 1):
        context_blocks.append(f"[{i}] (Page {item['page_index']}) {item['text']}")
    context = "\n\n".join(context_blocks)

    prompt = f"""Based on the context below, answer the user query.
Use ONLY the provided context to generate your answer.
Do NOT make up information or use outside knowledge.
If the answer cannot be found in the context, say "I don't know."
Be clear, accurate, and explanatory.

---

Context:
{context}

---

User Query:
{query}

Answer:"""

    return prompt


def generate_answer(
    prompt: str,
    tokenizer,
    model,
    max_new_tokens: int = 512,
    temperature: float = 0.7
) -> str:
    # Apply chat template
    full_prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize
    inputs = tokenizer(full_prompt, return_tensors="pt").to(device)
    input_length = inputs["input_ids"].shape[1]  # ← KEY FIX

    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True
        )

    # ← Slice from input_length instead of string replace
    new_tokens = outputs[0][input_length:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    return answer

## Section 16 — End-to-End RAG Pipeline

Now we tie everything together into a single `ask()` function:
**Retrieve → Augment → Generate**

In [ ]:
# -----------------------------------------------
# End-to-End RAG: ask() function
# -----------------------------------------------

def ask(
    query: str,
    index: faiss.IndexFlatIP,
    chunk_records: list[dict],
    embedding_type: str = "opensource",  # 'opensource' or 'cohere'
    k: int = 5,
    max_new_tokens: int = 512,
    temperature: float = 0.7,
    return_context: bool = False
):
    """
    Full RAG pipeline: retrieve → augment → generate.

    Parameters
    ----------
    query          : natural language question
    index          : FAISS index to search
    chunk_records  : list of chunk dicts matching the index
    embedding_type : 'opensource' or 'cohere'
    k              : number of chunks to retrieve
    max_new_tokens : max tokens for generation
    temperature    : generation temperature
    return_context : if True, also return retrieved chunks

    Returns
    -------
    answer string (and optionally context items)
    """

    # --- STEP 1: RETRIEVE ---
    if embedding_type == "opensource":
        context_items = retrieve_opensource(
            query, index, chunk_records, os_embedding_model, k
        )
    else:
        context_items = retrieve_cohere(
            query, index, chunk_records, co, k
        )

    # --- STEP 2: AUGMENT ---
    prompt = build_prompt(query, context_items)

    # --- STEP 3: GENERATE ---
    answer = generate_answer(prompt, tokenizer, llm_model, max_new_tokens, temperature)

    if return_context:
        return answer, context_items
    return answer


print("[INFO] ask() function defined. Ready for queries!")

[INFO] ask() function defined. Ready for queries!


## Section 17 — Run Sample Queries



In [ ]:
# -----------------------------------------------
# Sample Queries — Best Configuration
# Use sentence-aware + open-source as primary
# -----------------------------------------------

sample_queries = [
    # These match Buffett's actual writing style
    "What does Buffett say about Berkshire's ownership of businesses?",
    "How does Berkshire's insurance operation work?",
    "What does Buffett say about Berkshire's investments in stocks?",
    "How does Buffett describe Greg Abel's role at Berkshire?",
    "What does Buffett say about Berkshire's financial strength?",
]

# Run with sentence-aware chunks + open-source embeddings
for query in sample_queries:
    print("\n" + "=" * 80)
    print(f"QUERY: {query}")
    print("=" * 80)

    answer = ask(
        query=query,
        index=index_sentence_os,
        chunk_records=sentence_chunks,
        embedding_type="opensource",
        k=10
    )

    print("\nANSWER:")
    print(textwrap.fill(answer, 80))
    print()


QUERY: What does Buffett say about Berkshire's ownership of businesses?
[INFO] Retrieved 10 chunks in 0.00007 seconds

ANSWER:
Sure, here is the answer to the user query:  Buffett says that Berkshire's
ownership of businesses is characterized by its control and partial ownership of
various large and profitable businesses. With controlled companies, Berkshire
can dictate decisions, but has less flexibility in the disposition of mistakes.
With partial-ownership holdings, Berkshire has less control and cannot easily
change management or control capital flows.


QUERY: How does Berkshire's insurance operation work?
[INFO] Retrieved 10 chunks in 0.00006 seconds

ANSWER:
Sure, here is the answer to the user query:  Berkshire's insurance operation
works on a unique model where payment is received upfront but expenses are not
incurred simultaneously. This typically results in a mismatch between revenue
and cost, often leading to considerable cash haemorrhage. To account for this,
Berkshire's 

## Section 18 — Retrieval Quality Analysis

### Evaluation Framework
We evaluate our RAG system across three layers:
1. **Retrieval Quality** — Did we fetch the right evidence?
2. **Faithfulness** — Did the answer stay within retrieved context?
3. **Comparative Analysis** — Which chunking + embedding combo works best?

### Failure Mode Taxonomy
When a query fails, we classify the root cause:
- **Chunking failure** — answer was split across chunks
- **Embedding failure** — relevant text not mapped near the query
- **Ranking failure** — right chunks retrieved but scored too low
- **Prompting failure** — evidence present but LLM ignored it
- **Knowledge gap** — answer genuinely not in document

In [ ]:
# -----------------------------------------------
# Retrieval Quality Analysis
# Compare all 4 index combinations on same queries
# -----------------------------------------------

eval_queries = [
    "What does Buffett say about Berkshire's insurance operations?",
    "How does Buffett describe Berkshire's financial strength?",
    "What does Buffett say about Greg Abel?",
]

print("=" * 80)
print("RETRIEVAL QUALITY COMPARISON — Top Similarity Scores")
print("=" * 80)

results_summary = []

for query in eval_queries:
    print(f"\nQuery: {query}")
    print("-" * 60)

    # Index 1: Fixed + Open-source
    r1 = retrieve_opensource(query, index_fixed_os, fixed_chunks, os_embedding_model, k=3)
    avg1 = sum(r["score"] for r in r1) / len(r1)

    # Index 2: Fixed + Cohere
    r2 = retrieve_cohere(query, index_fixed_cohere, fixed_chunks, co, k=3)
    avg2 = sum(r["score"] for r in r2) / len(r2)

    # Index 3: Sentence + Open-source
    r3 = retrieve_opensource(query, index_sentence_os, sentence_chunks, os_embedding_model, k=3)
    avg3 = sum(r["score"] for r in r3) / len(r3)

    # Index 4: Sentence + Cohere
    r4 = retrieve_cohere(query, index_sentence_cohere, sentence_chunks, co, k=3)
    avg4 = sum(r["score"] for r in r4) / len(r4)

    print(f"  Fixed + Open-source    avg score: {avg1:.4f}")
    print(f"  Fixed + Cohere         avg score: {avg2:.4f}")
    print(f"  Sentence + Open-source avg score: {avg3:.4f}")
    print(f"  Sentence + Cohere      avg score: {avg4:.4f}")

    results_summary.append({
        "query": query,
        "fixed_os": avg1,
        "fixed_cohere": avg2,
        "sentence_os": avg3,
        "sentence_cohere": avg4
    })

# Summary table
summary_df = pd.DataFrame(results_summary)
print("\n" + "=" * 80)
print("OVERALL AVERAGE SCORES")
print("=" * 80)
print(f"Fixed + Open-source    : {summary_df['fixed_os'].mean():.4f}")
print(f"Fixed + Cohere         : {summary_df['fixed_cohere'].mean():.4f}")
print(f"Sentence + Open-source : {summary_df['sentence_os'].mean():.4f}")
print(f"Sentence + Cohere      : {summary_df['sentence_cohere'].mean():.4f}")
print("=" * 80)

RETRIEVAL QUALITY COMPARISON — Top Similarity Scores

Query: What does Buffett say about Berkshire's insurance operations?
------------------------------------------------------------
[INFO] Retrieved 3 chunks in 0.00007 seconds
[INFO] Retrieved 3 chunks in 0.00009 seconds
[INFO] Retrieved 3 chunks in 0.00006 seconds
[INFO] Retrieved 3 chunks in 0.00006 seconds
  Fixed + Open-source    avg score: 0.6968
  Fixed + Cohere         avg score: 0.6417
  Sentence + Open-source avg score: 0.7053
  Sentence + Cohere      avg score: 0.6207

Query: How does Buffett describe Berkshire's financial strength?
------------------------------------------------------------
[INFO] Retrieved 3 chunks in 0.00005 seconds
[INFO] Retrieved 3 chunks in 0.00006 seconds
[INFO] Retrieved 3 chunks in 0.00006 seconds
[INFO] Retrieved 3 chunks in 0.00006 seconds
  Fixed + Open-source    avg score: 0.6683
  Fixed + Cohere         avg score: 0.5773
  Sentence + Open-source avg score: 0.6767
  Sentence + Cohere      avg

## Section 19 — Faithfulness Analysis

We manually evaluate whether generated answers stay faithful to retrieved context.
For each answer we check: does every claim trace back to a retrieved chunk?

In [ ]:
# -----------------------------------------------
# Faithfulness Analysis
# Run a query and inspect context vs answer
# -----------------------------------------------

faithfulness_query = "How does Buffett describe his approach to managing risk?"

print("=" * 80)
print("FAITHFULNESS ANALYSIS")
print("=" * 80)
print(f"Query: {faithfulness_query}")
print()

# Get answer and context
answer, context_items = ask(
    query=faithfulness_query,
    index=index_sentence_os,
    chunk_records=sentence_chunks,
    embedding_type="opensource",
    k=10,
    return_context=True
)

# Display retrieved context
print("--- RETRIEVED CONTEXT ---")
for i, ctx in enumerate(context_items, 1):
    print(f"\n[Chunk {i}] Page {ctx['page_index']} | Score: {ctx['score']:.4f}")
    print(textwrap.fill(ctx["text"], 80))

# Display generated answer
print("\n" + "=" * 80)
print("--- GENERATED ANSWER ---")
print(textwrap.fill(answer, 80))

print("\n" + "=" * 80)
print("FAITHFULNESS CHECKLIST")
print("=" * 80)
print("Review the answer above against the retrieved chunks and ask:")
print("  [?] Does every claim in the answer appear in the retrieved chunks?")
print("  [?] Did the LLM add any information not present in the context?")
print("  [?] Did the LLM correctly say 'I don't know' for missing info?")

FAITHFULNESS ANALYSIS
Query: How does Buffett describe his approach to managing risk?

[INFO] Retrieved 10 chunks in 0.00007 seconds
--- RETRIEVED CONTEXT ---

[Chunk 1] Page 8 | Score: 0.5830
We make estimates for “surprises” and, so far, these estimates have been
sufficient. We are not deterred by the dramatic and growing loss payments
sustained by our  activities. (As I write this, think wildfires.) It’s our job
to price to absorb these and unemotionally  take our lumps when surprises
develop. It’s also our job to contest “runaway” verdicts, spurious  litigation
and outright fraudulent behavior. Under Ajit, our insurance operation has
blossomed from an obscure Omaha-based  company into a world leader, renowned for
both its taste for risk and its Gibraltar-like financial  strength. Moreover,
Greg, our directors and I all have a very large investment in Berkshire in
relation to any compensation we receive. We do not use options or other one-
sided forms of  compensation; if you lose m

## Section 20 — Failure Mode Analysis

We test edge cases to identify failure modes in our pipeline.

In [ ]:
# -----------------------------------------------
# Failure Mode Analysis
# Test queries likely to expose weaknesses
# -----------------------------------------------

failure_queries = [
    # Knowledge gap — this info is NOT in the 2024 letter
    "What did Buffett say about cryptocurrency in 2020?",

    # Ambiguous query — tests embedding quality
    "What is the main point?",

    # Very specific numeric fact — tests precision retrieval
    "What was Berkshire's exact operating earnings in 2024?",
]

print("=" * 80)
print("FAILURE MODE ANALYSIS")
print("=" * 80)

for query in failure_queries:
    print(f"\nQuery: {query}")
    print("-" * 60)

    answer, context = ask(
        query=query,
        index=index_sentence_os,
        chunk_records=sentence_chunks,
        embedding_type="opensource",
        k=3,
        return_context=True
    )

    print(f"Top retrieved chunk score: {context[0]['score']:.4f}")
    print(f"Answer: {textwrap.fill(answer, 70)}")
    print()

print("=" * 80)
print("FAILURE TAXONOMY GUIDE")
print("=" * 80)
print("""
Classify each failure above using these categories:

  CHUNKING FAILURE    — Answer was split across chunk boundaries
  EMBEDDING FAILURE   — Relevant text was not mapped near the query vector
  RANKING FAILURE     — Right chunks retrieved but scored too low
  PROMPTING FAILURE   — Evidence was present but LLM ignored or misused it
  KNOWLEDGE GAP       — Answer genuinely does not exist in the document
""")

FAILURE MODE ANALYSIS

Query: What did Buffett say about cryptocurrency in 2020?
------------------------------------------------------------
[INFO] Retrieved 3 chunks in 0.00006 seconds
Top retrieved chunk score: 0.5119
Answer: The text does not mention cryptocurrency or any related topic,
therefore I don't know the answer to the user query.


Query: What is the main point?
------------------------------------------------------------
[INFO] Retrieved 3 chunks in 0.00006 seconds
Top retrieved chunk score: 0.3524
Answer: The main point of the text is about the economic system called
capitalism and its impact on America's progress.


Query: What was Berkshire's exact operating earnings in 2024?
------------------------------------------------------------
[INFO] Retrieved 3 chunks in 0.00008 seconds
Top retrieved chunk score: 0.6973
Answer: Sure, here is the answer to the user query:  The text states that
Berkshire's operating earnings in 2024 were $47.4 billion.

FAILURE TAXONOMY GUIDE



## Section 21 — Final Summary & Conclusions

Summarize findings across all experiments.

In [ ]:
pages = len(pdf_pages)
words = stats_df['word_count'].sum()
fixed_count = len(fixed_chunks)
sentence_count = len(sentence_chunks)

print("=" * 80)
print("FINAL RAG SYSTEM SUMMARY REPORT")
print("=" * 80)

print(f"""
DOCUMENT
--------
  Source      : Warren Buffett 2024 Shareholder Letter
  Pages       : {pages}
  Total words : {words:,}

CHUNKING STRATEGIES
-------------------
  Strategy 1 - Fixed-Size with Overlap
    Chunk size  : 1000 characters
    Overlap     : 200 characters (20%)
    Total chunks: {fixed_count}

  Strategy 2 - Sentence-Aware
    Sentences/chunk: 10
    Total chunks   : {sentence_count}

EMBEDDING MODELS
----------------
  Open-source : all-mpnet-base-v2 (768 dimensions)
  Closed-source: Cohere embed-english-v3.0 (1024 dimensions)

VECTOR STORE
------------
  FAISS IndexFlatIP (cosine similarity)
  4 indexes built (2 strategies x 2 embedding models)

LLM
---
  Model    : Gemma-7B-IT
  Precision: Float16
  Hardware : A100 GPU (Colab Pro)

KEY FINDINGS
------------
- Avg tokens/page (678) nearly doubled the embedding model
  limit (384), confirming chunking is essential to prevent
  silent truncation of semantic content.
- Fixed-size chunking produced 106 chunks (avg 117 tokens),
  Sentence-aware produced 98 chunks (avg 102 tokens).
- Cohere embeddings consistently scored higher than
  open-source across all queries (avg 0.51 vs 0.49).
- Sentence-aware + Cohere achieved the highest avg
  retrieval score (0.5184) on finance-specific queries.
- Retrieval speed was extremely fast (~0.0001 seconds)
  confirming FAISS efficiency even with 4 separate indexes.
- Increasing k improves answer quality - small chunks
  require more retrieved context for coherent generation.

FAILURE MODES OBSERVED
----------------------
- EMBEDDING FAILURE: Vague query 'What is the main point?'
  scored only 0.3524 - the query lacked semantic signal
  for the embedding model to match against.
- PROMPTING FAILURE: Same query caused hallucination about
  'Carrie Sova' - LLM used outside knowledge despite
  grounding instructions when context was weak.
- KNOWLEDGE GAP: Cryptocurrency query correctly returned
  'I don't know' - model respected grounding constraints.
- RETRIEVAL FAILURE: Long-term investment queries returned
  'I don't know' despite relevant chunks being retrieved,
  suggesting chunk size (117 tokens) was too small to
  provide sufficient context for generation.

SCORE IMPROVEMENT ANALYSIS
--------------------------
Before optimization (chunk size 500, 5 sentences):
  Avg scores: 0.52 across all configurations

After optimization (chunk size 1000, 10 sentences):
  Fixed + Open-source    : 0.6147 (+17% improvement)
  Sentence + Open-source : 0.6187 (+17% improvement)

Key insight: Doubling chunk size had a much larger impact
on open-source embeddings (+17%) than Cohere (+4%),
suggesting Cohere's model is more robust to chunk size
variations due to its larger training corpus and
higher-dimensional vectors (1024 vs 768).
""")

print("=" * 80)
print("RAG PIPELINE COMPLETE ✅")
print("=" * 80)

FINAL RAG SYSTEM SUMMARY REPORT

DOCUMENT
--------
  Source      : Warren Buffett 2024 Shareholder Letter
  Pages       : 15
  Total words : 8,916

CHUNKING STRATEGIES
-------------------
  Strategy 1 - Fixed-Size with Overlap
    Chunk size  : 1000 characters
    Overlap     : 200 characters (20%)
    Total chunks: 57

  Strategy 2 - Sentence-Aware
    Sentences/chunk: 10
    Total chunks   : 57

EMBEDDING MODELS
----------------
  Open-source : all-mpnet-base-v2 (768 dimensions)
  Closed-source: Cohere embed-english-v3.0 (1024 dimensions)

VECTOR STORE
------------
  FAISS IndexFlatIP (cosine similarity)
  4 indexes built (2 strategies x 2 embedding models)

LLM
---
  Model    : Gemma-7B-IT
  Precision: Float16
  Hardware : A100 GPU (Colab Pro)

KEY FINDINGS
------------
- Avg tokens/page (678) nearly doubled the embedding model
  limit (384), confirming chunking is essential to prevent
  silent truncation of semantic content.
- Fixed-size chunking produced 106 chunks (avg 117 tokens

In [ ]:
# -----------------------------------------------
# Section 22 — Gradio Interface
# Interactive RAG Chatbot UI
# -----------------------------------------------
!pip install -q gradio

import gradio as gr

def rag_interface(query, embedding_choice, k_value):
    """
    Wrapper for the RAG pipeline to be used in Gradio.
    Allows user to select embedding type and k value.
    """
    try:
        # Select index based on embedding choice
        if embedding_choice == "Open-Source (all-mpnet-base-v2)":
            index = index_sentence_os
            chunks = sentence_chunks
            emb_type = "opensource"
        else:
            index = index_sentence_cohere
            chunks = sentence_chunks
            emb_type = "cohere"

        # Run RAG pipeline
        answer, context_items = ask(
            query=query,
            index=index,
            chunk_records=chunks,
            embedding_type=emb_type,
            k=int(k_value),
            return_context=True
        )

        # Format retrieved context for display
        context_display = ""
        for i, item in enumerate(context_items, 1):
            context_display += f"--- Source {i} ---\n"
            context_display += f"Page: {item['page_index']} | "
            context_display += f"Score: {item['score']:.4f}\n"
            context_display += f"{item['text']}\n\n"

        return answer, context_display

    except Exception as e:
        return f"Error: {str(e)}", ""


# Build Gradio Interface
with gr.Blocks(title="Berkshire RAG System", theme=gr.themes.Soft()) as demo:

    gr.Markdown("# 📄 Berkshire Hathaway RAG Chatbot")
    gr.Markdown(
        "Ask questions about Warren Buffett's **2024 Shareholder Letter**. "
        "The system retrieves relevant passages and generates grounded answers."
    )

    with gr.Row():
        with gr.Column(scale=2):
            query_input = gr.Textbox(
                label="Your Question",
                placeholder="e.g. What does Buffett say about Berkshire's insurance operations?",
                lines=2
            )

            with gr.Row():
                embedding_choice = gr.Radio(
                    choices=[
                        "Open-Source (all-mpnet-base-v2)",
                        "Closed-Source (Cohere)"
                    ],
                    value="Open-Source (all-mpnet-base-v2)",
                    label="Embedding Model"
                )
                k_slider = gr.Slider(
                    minimum=3,
                    maximum=15,
                    value=10,
                    step=1,
                    label="Top-K Chunks to Retrieve"
                )

            submit_btn = gr.Button("Ask RAG System", variant="primary")

            gr.Markdown("### 💬 Generated Answer")
            answer_output = gr.Textbox(
                label="Answer",
                lines=6,
                interactive=False
            )

        with gr.Column(scale=1):
            gr.Markdown("### 📚 Retrieved Sources")
            context_output = gr.Textbox(
                label="Retrieved Chunks",
                lines=20,
                interactive=False,
                show_copy_button=True
            )

    # Example queries
    gr.Examples(
        examples=[
            ["What does Buffett say about Berkshire's insurance operations?"],
            ["How does Buffett describe Berkshire's financial strength?"],
            ["What was Berkshire's exact operating earnings in 2024?"],
            ["What does Buffett say about Greg Abel?"],
            ["What did Buffett say about cryptocurrency in 2020?"],
        ],
        inputs=query_input
    )

    # Wire button
    submit_btn.click(
        fn=rag_interface,
        inputs=[query_input, embedding_choice, k_slider],
        outputs=[answer_output, context_output]
    )

# Launch
demo.launch(share=True, debug=True)

/tmp/ipykernel_6577/2741354812.py:50: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Berkshire RAG System", theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6d59e5b044dc66a9c3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[INFO] Retrieved 10 chunks in 0.00007 seconds
[INFO] Retrieved 10 chunks in 0.00007 seconds
[INFO] Retrieved 4 chunks in 0.00006 seconds
[INFO] Retrieved 15 chunks in 0.00007 seconds
[INFO] Retrieved 9 chunks in 0.00006 seconds
[INFO] Retrieved 9 chunks in 0.00008 seconds
